In [24]:
from utils.read_write_model import *


def w2c_to_c2w(image):
    R = qvec2rotmat(image.qvec)  # shape (3, 3)
    t = image.tvec  # shape (3, )

    c2w = np.eye(4)
    c2w[:3, :3] = R.T
    c2w[:3, 3] = -R.T @ t
    return c2w


In [5]:
from cumodel import NeuralPrimitiveSceneTileCUDA
from multiprocessing.util import info
import os
import uuid
from tqdm import tqdm
from datetime import datetime
import copy
import time
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
from vmodel import NeuralPrimitiveScene
# from vmodel import NeuralPrimitiveScene
from mip360_loader import mip360Dataset
from save_model import save_model
from pruning_densification import *



device = "cuda"
data_dir = "/home/yuzhou/Desktop/datasets/360_v2/bonsai"
dataset = mip360Dataset(data_dir, split="test", factor=4)
initial_num_primitives = 8436
initial_num_primitives = 2183
scene = NeuralPrimitiveSceneTileCUDA(
    num_primitives=initial_num_primitives,
    hidden_dim=32,
    pe_frequencies=6,
    initial_positions=dataset.positions,
).to(device)


model_ckpt_path = "/home/yuzhou/Desktop/output/mip360/8436/scene_epoch_98000.pt"
model_ckpt_path = "/home/yuzhou/Desktop/output/mip360/delta_b/scene_epoch_0.pt"
if os.path.exists(model_ckpt_path):
    ckpt = torch.load(model_ckpt_path, map_location=device)
    scene.load_state_dict(ckpt)
    print(f"[Checkpoint] Loaded model from {model_ckpt_path}")


def _build_projection_from_focal(focal: float, W: int, H: int, near: float = 0.001, far: float = 100.0, device: str | torch.device = "cpu"):
    """Match the viewer's perspective projection convention used by tile_cuda culling."""
    focal = float(focal)
    # vertical fov from focal length (pixels)
    fov_y = 2.0 * float(torch.atan(torch.tensor((0.5 * H) / (focal + 1e-8))))
    aspect = float(W) / float(H)
    t = float(torch.tan(torch.tensor(fov_y / 2.0)))
    r = t * aspect
    P = torch.zeros(4, 4, dtype=torch.float32, device=device)
    P[0, 0] = 1.0 / r
    P[1, 1] = 1.0 / t
    P[2, 2] = -(far + near) / (far - near)
    P[3, 2] = 1.0
    P[2, 3] = -(2 * far * near) / (far - near)
    return P


def _c2w_to_view_matrix(c2w_3x4: torch.Tensor, device: str | torch.device):
    """Convert (3,4) camera-to-world to (4,4) world-to-camera (view) matrix."""
    c2w_h = torch.eye(4, dtype=torch.float32, device=device)
    c2w_h[:3, :4] = c2w_3x4.to(device=device, dtype=torch.float32)
    view = torch.inverse(c2w_h)
    return view


psnr_all = 0

"""
for img_idx in tqdm(range(len(dataset))):
    # ---------------- Load data ----------------
    rays_o_hw, rays_d_hw, img_hw3, pose = dataset[img_idx]
    Himg, Wimg = img_hw3.shape[0], img_hw3.shape[1]
    scene.set_image_size(Wimg, Himg)

    rays_o_batch = rays_o_hw.reshape(-1, 3).to(device, non_blocking=True)
    rays_d_batch = rays_d_hw.reshape(-1, 3).to(device, non_blocking=True)
    rgbs_batch = img_hw3.reshape(-1, 3).to(device, non_blocking=True)

    view_matrix = _c2w_to_view_matrix(pose, device=device)
    proj_matrix = _build_projection_from_focal(dataset.focal, Wimg, Himg, device=device)


    pred = scene(
        rays_o_batch, rays_d_batch,
        score_threshold=1e-3,
        bg_color=torch.zeros(3, device=device),
        view_matrix=view_matrix,
        proj_matrix=proj_matrix,
    )

    psnr = -10 * torch.log10(torch.mean((pred - rgbs_batch) ** 2)).item()
    print(f"Image {img_idx}: PSNR = {psnr:.2f}")
    psnr_all += psnr

    import matplotlib.pyplot as plt
    pred_img = pred.cpu().view(1, 520, 780, 3)
    gt_img = rgbs_batch.cpu().view(1, 520, 780, 3)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax0_img = axes[0].imshow(gt_img[0].detach().numpy())
    axes[0].set_title("Ground Truth")
    ax1_img = axes[1].imshow(pred_img[0].detach().numpy())
    axes[1].set_title(f"Prediction (PSNR: {psnr:.2f} dB)")
    plt.tight_layout()
    plt.ion()
    plt.show()

psnr_final = psnr_all / len(dataset)

print(f"Final average PSNR over {len(dataset)} images: {psnr_final:.2f}")
"""

Loaded 37 images from /home/yuzhou/Desktop/datasets/360_v2/bonsai/images_4
Sampled 2183 initial positions from point cloud's 206613 points
[Checkpoint] Loaded model from /home/yuzhou/Desktop/output/mip360/delta_b/scene_epoch_0.pt


'\nfor img_idx in tqdm(range(len(dataset))):\n    # ---------------- Load data ----------------\n    rays_o_hw, rays_d_hw, img_hw3, pose = dataset[img_idx]\n    Himg, Wimg = img_hw3.shape[0], img_hw3.shape[1]\n    scene.set_image_size(Wimg, Himg)\n\n    rays_o_batch = rays_o_hw.reshape(-1, 3).to(device, non_blocking=True)\n    rays_d_batch = rays_d_hw.reshape(-1, 3).to(device, non_blocking=True)\n    rgbs_batch = img_hw3.reshape(-1, 3).to(device, non_blocking=True)\n\n    view_matrix = _c2w_to_view_matrix(pose, device=device)\n    proj_matrix = _build_projection_from_focal(dataset.focal, Wimg, Himg, device=device)\n\n\n    pred = scene(\n        rays_o_batch, rays_d_batch,\n        score_threshold=1e-3,\n        bg_color=torch.zeros(3, device=device),\n        view_matrix=view_matrix,\n        proj_matrix=proj_matrix,\n    )\n\n    psnr = -10 * torch.log10(torch.mean((pred - rgbs_batch) ** 2)).item()\n    print(f"Image {img_idx}: PSNR = {psnr:.2f}")\n    psnr_all += psnr\n\n    import 

In [6]:
import time
from tqdm import tqdm

# ---------------- Preload all frames ----------------
preloaded = []

for img_idx in range(len(dataset)):
    rays_o_hw, rays_d_hw, img_hw3, pose = dataset[img_idx]

    Himg, Wimg = img_hw3.shape[0], img_hw3.shape[1]

    rays_o_batch = rays_o_hw.reshape(-1, 3).to(device, non_blocking=True)
    rays_d_batch = rays_d_hw.reshape(-1, 3).to(device, non_blocking=True)
    rgbs_batch  = img_hw3.reshape(-1, 3).to(device, non_blocking=True)

    view_matrix = _c2w_to_view_matrix(pose, device=device)
    proj_matrix = _build_projection_from_focal(
        dataset.focal, Wimg, Himg, device=device
    )

    preloaded.append({
        "W": Wimg,
        "H": Himg,
        "rays_o": rays_o_batch,
        "rays_d": rays_d_batch,
        "rgbs": rgbs_batch,
        "view": view_matrix,
        "proj": proj_matrix,
    })

scene.set_image_size(preloaded[0]["W"], preloaded[0]["H"])



In [ ]:
import time, torch
frame = preloaded[0]
# warmup
for _ in range(10):
    _ = scene(frame["rays_o"], frame["rays_d"],
              score_threshold=1e-3,
              bg_color=torch.zeros(3, device=device),
              view_matrix=frame["view"], proj_matrix=frame["proj"])
zerotensor = torch.zeros(3, device=device)
torch.cuda.synchronize()

a = time.time()
for frame in preloaded:
    pred = scene(frame["rays_o"], frame["rays_d"],
                 score_threshold=1e-3,
                 bg_color=zerotensor,
                 view_matrix=frame["view"], proj_matrix=frame["proj"])
torch.cuda.synchronize()

dt = time.time() - a
print(f"Total inference time for {len(preloaded)} frames: {dt:.5f} s")
print(f"Average FPS: {len(preloaded)/dt:.2f}")


Total inference time for 37 frames: 4.55905 s
Average FPS: 8.12


In [25]:
model_path = "datasets/nerf_llff_data/fern/sparse/0/"  # Replace with your actual path

# Read the binary files
cameras = read_cameras_binary(f"{model_path}/cameras.bin")
images = read_images_binary(f"{model_path}/images.bin")
points3D = read_points3D_binary(f"{model_path}/points3D.bin")

# Example: print number of each component
print(f"Loaded {len(cameras)} cameras")
print(f"Loaded {len(images)} images")
print(f"Loaded {len(points3D)} 3D points")

# 只打印第一个图像的位姿信息
image_id, image = next(iter(images.items()))
print(f"\nImage ID: {image_id}")
print(f"Name: {image.name}")
print(f"Camera ID: {image.camera_id}")
print(f"Qvec (wxyz): {image.qvec}")
print(f"Tvec: {image.tvec}")
print(f"2D points shape: {image.xys.shape}")


print(w2c_to_c2w(image))

data_dir = "/home/yuzhou/Desktop/mlp-splatting/datasets/nerf_llff_data/fern"
poses_bounds = np.load(os.path.join(data_dir, "poses_bounds.npy"))
poses = poses_bounds[:, :-2].reshape([-1, 3, 5])[:, :, :4]

poses[0]

Loaded 1 cameras
Loaded 20 images
Loaded 10091 3D points

Image ID: 1
Name: IMG_4026.JPG
Camera ID: 1
Qvec (wxyz): [ 0.99726159  0.01687281 -0.07092128  0.01244197]
Tvec: [3.55835243 1.67507458 0.84698212]
2D points shape: (11187, 2)
[[ 0.98963074  0.02242252  0.141874   -3.67917907]
 [-0.02720908  0.99912101  0.0318884  -1.60379162]
 [-0.14103427 -0.035418    0.98937096 -0.27680206]
 [ 0.          0.          0.          1.        ]]


array([[ 0.02242252,  0.98963074, -0.141874  , -3.67917907],
       [ 0.99912101, -0.02720908, -0.0318884 , -1.60379162],
       [-0.035418  , -0.14103427, -0.98937096, -0.27680206]])

In [19]:
poses_bounds[:, :-2].shape

(20, 15)

In [18]:
poses_bounds.shape

(20, 17)

In [8]:
# 只打印第一个图像的位姿信息
image_id, image = next(iter(images.items()))
print(f"\nImage ID: {image_id}")
print(f"Name: {image.name}")
print(f"Camera ID: {image.camera_id}")
print(f"Qvec (wxyz): {image.qvec}")
print(f"Tvec: {image.tvec}")
print(f"2D points shape: {image.xys.shape}")


Image ID: 1
Name: IMG_4026.JPG
Camera ID: 1
Qvec (wxyz): [ 0.99726159  0.01687281 -0.07092128  0.01244197]
Tvec: [3.55835243 1.67507458 0.84698212]
2D points shape: (11187, 2)


In [ ]:
"""
# Example: access camera intrinsics
for cam_id, cam in cameras.items():
    print(f"\nCamera ID: {cam_id}")
    print(f"Model: {cam.model}")
    print(f"Width x Height: {cam.width} x {cam.height}")
    print(f"Params: {cam.params}")  # fx, fy, cx, cy, etc.

# Example: print 3D point cloud
for point_id, pt in points3D.items():
    print(f"\n3D Point ID: {point_id}")
    print(f"XYZ: {pt.xyz}")
    print(f"Color: {pt.rgb}")
    print(f"Error: {pt.error}")
    print(f"Track: {pt.image_ids}")
    if point_id > 5:  # Print only a few points
        break
"""

In [4]:
import torch
torch.log(torch.tensor(0.1))  # 注意使用 float，否则结果是 -inf
torch.exp(torch.tensor(-2.2594))

tensor(0.1044)